In [11]:
# Week 7 Assignment
# Retrieval-Augmented Generation (RAG) System

## Introduction

# Retrieval-Augmented Generation (RAG) is a framework that combines information retrieval with Large Language Models (LLMs) to generate accurate and context-aware responses. Instead of relying only on the knowledge stored within the language model, RAG first retrieves relevant information from external documents and then uses that retrieved context to produce an answer.

# In this project, a complete RAG pipeline is developed using the Hugging Face RAG Mini Wikipedia dataset. The pipeline performs document ingestion, text chunking, embedding generation, vector storage using FAISS, semantic retrieval, and answer generation using Google's FLAN-T5 model.

In [12]:
# Aim

# The objective of this assignment is to build a Retrieval-Augmented Generation (RAG) system capable of answering user queries using custom documents.

# The system performs the following tasks:

# - Load custom documents.
# - Split documents into smaller chunks.
# - Generate dense vector embeddings.
# - Store embeddings in a FAISS vector database.
# - Retrieve the most relevant chunks for a query.
# - Generate grounded answers using an LLM.
# - Evaluate the retrieval performance.

In [1]:
# SetUp
!pip install -q datasets sentence-transformers faiss-cpu transformers rank_bm25 accelerate

import numpy as np
import pandas as pd
import textwrap, time, warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi

print("Environment Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 32.0 MB/s eta 0:00:00
Environment Ready!


In [2]:
# Document Ingestion
def load_documents():
    ds = load_dataset(
        "rag-datasets/rag-mini-wikipedia",
        "text-corpus",
        split="passages"
    )
    docs = [row["passage"] for row in ds if row["passage"].strip()]
    print(f"Loaded {len(docs)} passages.")
    return docs

documents = load_documents()

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet:   0%|          | 0.00/797k [00:00<?, ?B/s]

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

Loaded 3200 passages.


In [3]:
# Chunking
def clean_text(text):
    return " ".join(text.split())

def split_chunks(text, size=300, overlap=50):
    text = clean_text(text)
    chunks = []
    step = size - overlap
    for i in range(0, len(text), step):
        part = text[i:i+size]
        if part:
            chunks.append(part)
    return chunks

all_chunks=[]
source_ids=[]
for i,d in enumerate(documents):
    c = split_chunks(d)
    all_chunks.extend(c)
    source_ids.extend([i]*len(c))

print("Chunks:",len(all_chunks))

Chunks: 6745


In [4]:
# Embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(
    all_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

dimension = embeddings.shape[1]
print("Embedding Shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/211 [00:00<?, ?it/s]

Embedding Shape: (6745, 384)


In [5]:
# FAISS

index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype("float32"))
print("Vectors Stored:", index.ntotal)

Vectors Stored: 6745


In [6]:
# Retriever

def search(query, top_k=3):
    q = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(q, top_k)

    results=[]
    for score,idx in zip(scores[0],ids[0]):
        results.append({
            "text":all_chunks[idx],
            "score":float(score)
        })
    return results

print(search("What is machine learning?")[0])

{'text': ' machine. The stored information could then be processed through an artificial optic nerve and played back as visual patterns on a viewscreen.', 'score': 0.3451853394508362}


In [7]:
# Hybrid Search
bm25 = BM25Okapi([c.lower().split() for c in all_chunks])

def hybrid_search(query, top_k=3, alpha=0.6):
    q = embedder.encode([query], convert_to_numpy=True,
                        normalize_embeddings=True)
    vector = (embeddings @ q.T).flatten()
    keyword = np.array(bm25.get_scores(query.lower().split()))

    vector = (vector-vector.min())/(vector.max()-vector.min()+1e-9)
    keyword=(keyword-keyword.min())/(keyword.max()-keyword.min()+1e-9)

    score = alpha*vector + (1-alpha)*keyword
    idx=np.argsort(score)[::-1][:top_k]
    return [all_chunks[i] for i in idx]

In [8]:
# Generation

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name="google/flan-t5-base"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

device="cuda" if torch.cuda.is_available() else "cpu"
model=model.to(device)

def answer(query):
    context="\n".join(hybrid_search(query))
    prompt=f"Context:\n{context}\n\nQuestion:{query}\nAnswer:"
    x=tokenizer(prompt,return_tensors="pt",
                truncation=True,max_length=512).to(device)
    y=model.generate(**x,max_new_tokens=80)
    return tokenizer.decode(y[0],skip_special_tokens=True)

print(answer("What is the capital of France?"))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Strasbourg


In [9]:
# Conclusion
print("Modified RAG pipeline executed successfully.")

Modified RAG pipeline executed successfully.
